# JN04 — Building a Bathymetric Grid for Tsunami-HySEA from GEBCO
## Single-Level Grid — No Nesting Required

**HySEALab · Preprocessing Notebooks · EDANYA Research Group, Universidad de Málaga**
*Edited by José Manuel González Vida*

This notebook covers the **simplest possible workflow** to create a
bathymetric grid file (`.grd`) ready for Tsunami-HySEA, using only
the freely available **GEBCO** global dataset.

No nested grids, no high-resolution regional data, no GMT tools — just
Python + the GEBCO GeoTIFF.

This notebook is **self-contained**: it only requires the Python packages
listed below and a GEBCO GeoTIFF tile covering your study area.

---

### Requirements

**Python packages** (any recent version):

```bash
conda install -c conda-forge numpy scipy matplotlib pillow netcdf4
# or: pip install numpy scipy matplotlib pillow netCDF4
```

**Input data — GEBCO GeoTIFF:**

1. Go to the GEBCO download tool: [download.gebco.net](https://download.gebco.net)
2. Select a region that fully covers your study area
3. Choose format **2D GeoTIFF** (grid: *GEBCO sub-ice topo/bathy* is recommended)
4. Place the downloaded `.tif` file in the `data/` folder next to this notebook
5. Set `GEBCO_PATH` in Step 1 to the file name

> Any GEBCO release (2023, 2024, 2025…) and any tile size works —
> the notebook reads the georeferencing directly from the file.

---

### What we will do

```
GEBCO  (GeoTIFF tile, possibly hundreds of MB)
       │
       ▼  Step 1 — Read metadata (pixel size, extent)
       │
       ▼  Step 2 — Define the study area
       │
       ▼  Step 3 — Clip and resample to the desired resolution
       │
       ▼  Step 4 — Write the HySEA .grd file (NetCDF4)
       │
       ▼  Step 5 — Verify: plot + check dimensions
       │
       ▼  Step 6 — Reference in the HySEA parameter file
```

### Files used
| File | Description |
|------|-------------|
| `data/gebco_2025_sub_ice_n90.0_s0.0_w0.0_e90.0.tif` | GEBCO GeoTIFF tile (example: 2025 release, quadrant [0–90°E, 0–90°N]) |

### Output
| File | Description |
|------|-------------|
| `data/GEBCO_emed_L0.grd` | Ready-to-use HySEA bathymetric grid |
| `data/parfile_GEBCO_emed.txt` | Template Tsunami-HySEA parameter file referencing the grid |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from netCDF4 import Dataset
from PIL import Image
Image.MAX_IMAGE_PIXELS = None   # GEBCO is large — disable PIL safety limit
from scipy.interpolate import RegularGridInterpolator
from datetime import datetime
import os

DATA_DIR = 'data'               # input/output folder, relative to this notebook
os.makedirs(DATA_DIR, exist_ok=True)

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size']  = 11

print('Libraries OK')
print(f'Working directory: {os.getcwd()}')
print(f'Data directory   : {os.path.abspath(DATA_DIR)}')

---
## Step 1 — Read GEBCO Metadata

**What is GEBCO?**
The General Bathymetric Chart of the Oceans (GEBCO) is the global standard
for seafloor depth. The 2025 edition has a resolution of **15 arc-seconds ≈ 460 m**
and is freely downloadable at [www.gebco.net](https://www.gebco.net).

**Sign convention:** GEBCO uses the standard GMT convention:
- Negative values = ocean depth (e.g. −3000 m)
- Positive values = land elevation (e.g. +200 m)

The HySEA `.grd` format uses **the same convention** — so no sign flip needed.

**Reading georeferencing without rasterio:**
A GeoTIFF stores its coordinate system in TIFF tags:
- Tag **33550** (`ModelPixelScaleTag`) → pixel size in degrees `(dx, dy)`
- Tag **33922** (`ModelTiepointTag`) → coordinates of the top-left corner `(lon_UL, lat_UL)`

In [ ]:
# ── USER PARAMETER ──────────────────────────────────────────────────────────
#   Name of the GEBCO GeoTIFF tile you downloaded (see Requirements above)
GEBCO_PATH = os.path.join(DATA_DIR,
    'gebco_2025_sub_ice_n90.0_s0.0_w0.0_e90.0.tif')
# ────────────────────────────────────────────────────────────────────────────

if not os.path.exists(GEBCO_PATH):
    raise FileNotFoundError(
        f'GEBCO file not found: {GEBCO_PATH}\n'
        'Download a GeoTIFF tile covering your study area from '
        'https://download.gebco.net and place it in the data/ folder, '
        'then update GEBCO_PATH above.')

# Read only the header — do NOT load the full array yet
img = Image.open(GEBCO_PATH)
tv2 = img.tag_v2
width, height = img.size

# Extract pixel scale and tiepoint
dx_src = tv2[33550][0]   # degrees per pixel (longitude)
dy_src = tv2[33550][1]   # degrees per pixel (latitude, positive)
lon_ul = tv2[33922][3]   # longitude of upper-left pixel centre
lat_ul = tv2[33922][4]   # latitude  of upper-left pixel centre
img.close()

lon_lr = lon_ul + (width  - 1) * dx_src
lat_lr = lat_ul - (height - 1) * dy_src

print('GEBCO — file metadata')
print(f'  Grid size   : {width} × {height} pixels')
print(f'  Pixel size  : dx = {dx_src:.6f}°  dy = {dy_src:.6f}°')
print(f'              : dx ≈ {dx_src * 111320:.0f} m   dy ≈ {dy_src * 111320:.0f} m')
print(f'  Lon extent  : {lon_ul:.4f}° – {lon_lr:.4f}°E')
print(f'  Lat extent  : {lat_lr:.4f}° – {lat_ul:.4f}°N')
print(f'  File size   : {os.path.getsize(GEBCO_PATH)/1024**2:.0f} MB')

---
## Step 2 — Define the Study Area

**Key rule:** Only load the pixels you actually need.
A GEBCO tile can cover a large fraction of the globe — loading it
entirely could use several GB of RAM. Instead we compute the pixel
indices of our target bounding box and load only that window.

As a worked example we choose the **Eastern Mediterranean** basin
(Sicily — Ionian Sea), a classical tsunami propagation test area.

> **Exercise:** Change `LON_MIN`, `LON_MAX`, `LAT_MIN`, `LAT_MAX`
> to your own study area. Make sure it is contained within
> the GEBCO file extent shown above.

In [ ]:
# ── USER PARAMETERS ─────────────────────────────────────────────────────────
#   Change these to define your own study area

LON_MIN = 13.40   # degrees East
LON_MAX = 19.20
LAT_MIN = 34.70   # degrees North
LAT_MAX = 38.95

DX_TARGET = 0.004   # target resolution in degrees (~445 m)
                    # GEBCO native is 0.00417° — use multiples: 0.004, 0.008, 0.016 ...
                    # Smaller = finer grid = more RAM and longer simulation

OUTPUT_FILE = os.path.join(DATA_DIR, 'GEBCO_emed_L0.grd')
# ────────────────────────────────────────────────────────────────────────────

# Snap domain to DX_TARGET so the grid is aligned
lon_min_s = round(LON_MIN / DX_TARGET) * DX_TARGET
lon_max_s = round(LON_MAX / DX_TARGET) * DX_TARGET
lat_min_s = round(LAT_MIN / DX_TARGET) * DX_TARGET
lat_max_s = round(LAT_MAX / DX_TARGET) * DX_TARGET

nx = round((lon_max_s - lon_min_s) / DX_TARGET) + 1
ny = round((lat_max_s - lat_min_s) / DX_TARGET) + 1

print('Study area (snapped to grid):')
print(f'  Longitude : {lon_min_s:.4f}° – {lon_max_s:.4f}°E')
print(f'  Latitude  : {lat_min_s:.4f}° – {lat_max_s:.4f}°N')
print(f'  Resolution: {DX_TARGET}°  ≈  {DX_TARGET * 111320:.0f} m')
print(f'  Grid size : {nx} × {ny} = {nx*ny:,} cells')
print(f'  Memory    : ≈ {nx * ny * 4 / 1024**2:.1f} MB (float32)')

---
## Step 3 — Clip and Resample

We:
1. Compute which pixel rows/columns of GEBCO correspond to our bounding box
2. Load **only that window** using `PIL.Image.crop()`
3. Build a `RegularGridInterpolator` from the clipped source data
4. Interpolate onto the target regular grid at `DX_TARGET` resolution

> Loading the GEBCO window requires reading the source file from disk —
> for a tile of several hundred MB this can take ~20–30 seconds.
> The interpolation itself is fast.

In [ ]:
print('Clipping GEBCO to study area...')
print('(reading the source tile from disk — this may take ~20-30 s)')

img = Image.open(GEBCO_PATH)
tv2 = img.tag_v2
w, h = img.size
dx_s = tv2[33550][0];  dy_s = tv2[33550][1]
lon0 = tv2[33922][3];  lat0 = tv2[33922][4]

# Pixel indices of the bounding box (add 2-pixel margin)
col0 = max(0, int((LON_MIN - lon0) / dx_s) - 2)
col1 = min(w, int((LON_MAX - lon0) / dx_s) + 3)
row0 = max(0, int((lat0 - LAT_MAX) / dy_s) - 2)
row1 = min(h, int((lat0 - LAT_MIN) / dy_s) + 3)

# Crop — only this window is loaded into RAM
region = img.crop((col0, row0, col1, row1))
data_src = np.array(region, dtype=np.float32)
img.close()

# Reconstruct coordinate axes for the clipped window
lons_src = lon0 + np.arange(col0, col1) * dx_s
lats_src = lat0 - np.arange(row0, row1) * dy_s

# Flip to ascending latitude (required by RegularGridInterpolator)
lats_src = lats_src[::-1]
data_src = data_src[::-1, :]

print(f'Clipped window : {len(lons_src)} × {len(lats_src)} pixels')
print(f'Lon            : {lons_src[0]:.4f} – {lons_src[-1]:.4f}°E')
print(f'Lat            : {lats_src[0]:.4f} – {lats_src[-1]:.4f}°N')
print(f'Depth range    : {data_src.min():.0f} – {data_src.max():.0f} m')

In [ ]:
# Build target grid
lons_out = np.arange(lon_min_s, lon_max_s + DX_TARGET / 2, DX_TARGET)
lats_out = np.arange(lat_min_s, lat_max_s + DX_TARGET / 2, DX_TARGET)

print('Resampling to target grid...')
interp = RegularGridInterpolator(
    (lats_src, lons_src), data_src,
    method='linear', bounds_error=False, fill_value=np.nan)

LON_G, LAT_G = np.meshgrid(lons_out, lats_out)
pts = np.column_stack((LAT_G.ravel(), LON_G.ravel()))
z_out = interp(pts).reshape(len(lats_out), len(lons_out)).astype(np.float32)

print(f'Output grid : {z_out.shape}  ({len(lons_out)} × {len(lats_out)})')
print(f'Depth range : {np.nanmin(z_out):.0f} – {np.nanmax(z_out):.0f} m')

# ── Fill NaN gaps so Tsunami-HySEA never sees a hole ─────────────────────────
# NaN appears wherever the (snapped) target grid falls OUTSIDE the GEBCO source
# window — usually because the requested domain extends beyond the GEBCO tile.
# HySEA cannot run with NaN in the bathymetry, so we (1) report the mismatch and
# (2) fill the gaps by nearest-neighbour.
n_nan = int(np.isnan(z_out).sum())
if n_nan:
    frac = 100.0 * n_nan / z_out.size
    print(f'\n\u26a0  {n_nan} NaN cells ({frac:.2f}%) \u2014 target grid extends beyond the GEBCO data:')
    print(f'   GEBCO source : lon {lons_src[0]:.3f} \u2013 {lons_src[-1]:.3f}\u00b0E, '
          f'lat {lats_src[0]:.3f} \u2013 {lats_src[-1]:.3f}\u00b0N')
    print(f'   Target domain: lon {lons_out[0]:.3f} \u2013 {lons_out[-1]:.3f}\u00b0E, '
          f'lat {lats_out[0]:.3f} \u2013 {lats_out[-1]:.3f}\u00b0N')
    nn = RegularGridInterpolator((lats_src, lons_src), data_src,
                                 method='nearest', bounds_error=False, fill_value=None)
    mask = np.isnan(z_out)
    z_out[mask] = nn(pts[mask.ravel()]).astype(np.float32)
    still = int(np.isnan(z_out).sum())
    print(f'   \u2192 filled by nearest-neighbour; remaining NaN: {still}')
    print('   NOTE: filled cells are EXTRAPOLATED (nearest valid value). For a grid')
    print('   without synthetic data, shrink the domain to the source coverage above')
    print('   or use a GEBCO tile that fully covers your study area.')
else:
    print('NaN cells   : 0  \u2713')

In [ ]:
# ── Preview: visualize the resampled bathymetry before writing to disk ──
fig, ax = plt.subplots(figsize=(11, 7))

pcm = ax.pcolormesh(lons_out, lats_out, z_out,
                    cmap='terrain', shading='auto', vmin=-5000, vmax=500)
cb = plt.colorbar(pcm, ax=ax, label='Depth / Elevation (m)', shrink=0.85)

# Zero-contour ≈ coastline
cs0 = ax.contour(lons_out, lats_out, z_out, levels=[0],
                 colors='black', linewidths=1.0)

# Key isobaths
cs1 = ax.contour(lons_out, lats_out, z_out,
                 levels=[-3000, -2000, -1000, -500, -100],
                 colors='steelblue', linewidths=0.5, linestyles='--', alpha=0.7)
ax.clabel(cs1, fmt='%d m', fontsize=7, inline=True)

ax.set_xlabel('Longitude (°E)')
ax.set_ylabel('Latitude (°N)')
ax.set_title(
    f'Clipped & resampled GEBCO 2025  ·  '
    f'$\\Delta x$ = {DX_TARGET}°  $\\approx$  {DX_TARGET*111320:.0f} m\n'
    f'Grid: {len(lons_out)} × {len(lats_out)} cells  ·  '
    f'Depth range: {float(np.nanmin(z_out)):.0f} – {float(np.nanmax(z_out)):.0f} m',
    fontsize=11)
ax.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()
print('Preview OK — proceed to write the .grd file')


---
## Step 4 — Write the HySEA `.grd` File

The HySEA `.grd` format is a **NetCDF4** file with exactly three variables:

| Variable | Shape | Type | Description |
|----------|-------|------|-------------|
| `x` | `(nx,)` | float64 | Longitude array |
| `y` | `(ny,)` | float64 | Latitude array |
| `z` | `(ny, nx)` | float32 | Depth/elevation (m), **negative = ocean** |

The latitude array must be **ascending** (south to north).

In [ ]:
def grdwrite(x, y, z, foutput):
    """Write a Tsunami-HySEA compatible .grd file (NetCDF4)."""
    # Safety net: never write a grid with NaN/inf — HySEA would fail at runtime.
    z = np.asarray(z, dtype=np.float32)
    n_bad = int((~np.isfinite(z)).sum())
    if n_bad:
        raise ValueError(
            f'Refusing to write {os.path.basename(foutput)}: z contains {n_bad} '
            'non-finite cell(s) (NaN/inf). Fix the gaps in the resampling step '
            'before writing (see the nearest-neighbour fill above).')
    ds = Dataset(foutput, 'w', format='NETCDF4')
    ds.createDimension('x', len(x))
    ds.createDimension('y', len(y))
    vx = ds.createVariable('x', 'f8', 'x')
    vy = ds.createVariable('y', 'f8', 'y')
    vz = ds.createVariable('z', 'f4', ('y', 'x'))
    vx[:] = x;  vy[:] = y;  vz[:, :] = z
    vx.units = 'degrees_east'
    vy.units = 'degrees_north'
    vz.units = 'meters'
    ds.title       = os.path.basename(foutput)
    ds.history     = 'Created by JN04_Grid_from_GEBCO.ipynb'
    ds.description = 'GEBCO 2025 · ' + datetime.today().strftime('%d/%m/%Y')
    ds.close()
    size_mb = os.path.getsize(foutput) / 1024**2
    print(f'Written: {foutput}   ({size_mb:.1f} MB)')


grdwrite(lons_out, lats_out, z_out, OUTPUT_FILE)

---
## Step 5 — Verify the Grid

Before using any grid in a simulation, always verify:
- **Correct extent** — does the bounding box match what you defined?
- **Correct resolution** — is `dx` exactly what the parameter file expects?
- **No NaN** — gaps cause HySEA to crash
- **Sign convention** — ocean cells must be negative
- **Realistic depths** — plot to catch flipped data or wrong region

In [ ]:
def grdread(grdfile):
    """Read a HySEA .grd file. Returns (x, y, z)."""
    ds = Dataset(grdfile, mode='r')
    for vn in ds.variables:
        if vn in ('lon', 'x', 'longitude'): x = ds.variables[vn][:]
        if vn in ('lat', 'y', 'latitude'):  y = ds.variables[vn][:]
        if vn in ('topo', 'z', 'Band1'):    z = ds.variables[vn][:]
    ds.close()
    return x, y, z


xv, yv, zv = grdread(OUTPUT_FILE)
zv_np = np.array(zv)

print('VERIFICATION')
print('=' * 50)
print(f'  File         : {os.path.basename(OUTPUT_FILE)}')
print(f'  nx × ny      : {len(xv)} × {len(yv)}')
print(f'  Lon range    : {float(xv[0]):.4f} – {float(xv[-1]):.4f} °E')
print(f'  Lat range    : {float(yv[0]):.4f} – {float(yv[-1]):.4f} °N')
print(f'  dx           : {float(xv[1]-xv[0]):.6f}°  ≈ {float((xv[1]-xv[0])*111320):.0f} m')
print(f'  Depth range  : {float(np.nanmin(zv)):.0f} – {float(np.nanmax(zv)):.0f} m')
print(f'  NaN cells    : {int(np.isnan(zv_np).sum())}  ✓' if np.isnan(zv_np).sum()==0 else '  WARNING: NaN cells found')
ocean_pct = 100 * float(np.sum(zv_np < 0)) / zv_np.size
print(f'  Ocean cells  : {ocean_pct:.1f}%  (z < 0 → ocean, GMT convention)')

In [ ]:
# Plot the grid
fig, axes = plt.subplots(1, 2, figsize=(15, 6), constrained_layout=True)
fig.suptitle(
    f'GEBCO 2025 → {os.path.basename(OUTPUT_FILE)}\n'
    f'{len(xv)} × {len(yv)} cells  ·  Δx = {DX_TARGET}° ≈ {DX_TARGET*111320:.0f} m',
    fontsize=12)

cmap = plt.get_cmap('terrain').copy()

# Left: full bathymetric map
ax = axes[0]
im = ax.pcolormesh(xv, yv, zv, cmap=cmap, shading='auto', vmin=-4000, vmax=1000)
fig.colorbar(im, ax=ax, pad=0.02, label='Depth / Elevation (m)')
# Coastline (0-m contour)
ax.contour(xv, yv, zv, levels=[0], colors='black', linewidths=0.8)
# Isobath lines
cs = ax.contour(xv, yv, zv, levels=[-2000, -1000, -500, -200],
                colors='white', linewidths=0.4, linestyles='--', alpha=0.6)
ax.clabel(cs, fmt='%d m', fontsize=6, inline=True)
ax.set_xlabel('Longitude (°E)');  ax.set_ylabel('Latitude (°N)')
ax.set_title('Bathymetric map', fontsize=10)
ax.grid(True, linestyle='--', alpha=0.4)

# Earthquake source (Messina 1908 / Ionian Sea) — adjust as needed
src_lon, src_lat = 15.63, 37.91
ax.plot(src_lon, src_lat, '*', ms=14, color='gold', mec='black', mew=0.8,
        label='Source', zorder=5)
ax.legend(fontsize=9)

# Right: depth histogram (ocean only)
ax2 = axes[1]
ocean_vals = zv_np[zv_np < 0].ravel()
ax2.hist(ocean_vals, bins=60, color='steelblue', edgecolor='none', alpha=0.85)
ax2.axvline(float(np.median(ocean_vals)), color='red', lw=1.5,
            label=f'Median = {float(np.median(ocean_vals)):.0f} m')
ax2.set_xlabel('Depth (m, ocean cells only)')
ax2.set_ylabel('Count')
ax2.set_title('Depth distribution', fontsize=10)
ax2.legend(fontsize=9)
ax2.grid(True, linestyle='--', alpha=0.4)

plt.show()

---
## Step 6 — Reference the Grid in the HySEA Parameter File

Once the `.grd` file is verified, reference it in the parameter file.
Below we generate a minimal **single-level** parameter file template for a
propagation run (no inundation, no nested grids), using an Okada fault
source corresponding to a 1908 Messina-type earthquake scenario —
**adjust the source parameters to your own case**.

If you already have a reference parameter file, place it at
`data/parfile_prop.txt` and the cell below will print it for comparison
(this step is optional and is skipped if the file is not present).

In [ ]:
# Optional: show a reference parfile for comparison (skipped if not present)
ref_parfile = os.path.join(DATA_DIR, 'parfile_prop.txt')

if os.path.exists(ref_parfile):
    print('Reference parfile (parfile_prop.txt):')
    print('─' * 60)
    with open(ref_parfile) as f:
        lines = f.readlines()
    for line in lines:
        print(line, end='')
    print()
    print('─' * 60)
    print(f'  Line 2 — bathymetry file: {lines[1].strip()}')
    print()

# Generate a new parameter file template for our grid
# Source: Okada fault parameters for a 1908 Messina-type scenario — EDIT THESE
new_parfile = os.path.join(DATA_DIR, 'parfile_GEBCO_emed.txt')
template = f"""GEBCO_EMED_L0
{os.path.basename(OUTPUT_FILE)}
1
0
1
# Time Lon Lat Depth(km) Length(km) Width(km) Strike Dip Rake Slip(m)
0.0 15.63 37.91 16.25 204.9 61.0 22.5 30.0 90.0 4.21
0
simulations/GEBCO_emed_L0
1 1 0 0 0 0 0 0 0 1
1
1
1
1
1
7200.0
60
0
0.5
5e-3
20
0.2
0
0.03
100
10000
100
0.05
"""

with open(new_parfile, 'w') as f:
    f.write(template)

print(f'Template parfile written: {new_parfile}')
print()
print('To run HySEA:')
print(f'  TsunamiHySEA {os.path.basename(new_parfile)}')

---
## Summary — What We Did

| Step | Tool | Input → Output |
|------|------|----------------|
| Read metadata | `PIL.Image.open()` | `.tif` → pixel size, extent |
| Clip to domain | `PIL.Image.crop()` | full tile → small window |
| Resample | `scipy.RegularGridInterpolator` | native res → target res |
| Write `.grd` | `netCDF4.Dataset` | arrays → HySEA file |
| Verify | `grdread()` + `matplotlib` | plot + dimension check |
| Parameter file | text template | `.grd` reference + source params |

### Key settings to remember

```python
LON_MIN, LON_MAX = ...     # bounding box in degrees East
LAT_MIN, LAT_MAX = ...     # bounding box in degrees North
DX_TARGET = 0.004          # resolution in degrees
                           # GEBCO native ≈ 0.00417° (15 arc-sec)
                           # Use 0.004, 0.008, 0.016 ... for uniform grids
```

### When to add nested grids

| Objective | Min resolution needed | Use nesting? |
|-----------|----------------------|--------------|
| Ocean propagation + FCPs | 2–5 km (0.018–0.045°) | No |
| Coastal arrival times | 200–500 m (0.002–0.004°) | Optional |
| Inundation mapping | < 50 m (< 0.0005°) | **Yes** |

For inundation studies, a dedicated preprocessing notebook on **nested grid
construction** will be released in this collection — check the
`preprocessing/` folder of the HySEALab repository for updates.